<a href="https://colab.research.google.com/github/justii543/MLpreps/blob/main/CodeVulnerability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [1]:
!pip install transformers torch datasets pandas numpy scikit-learn

In [2]:
!git clone https://github.com/DLVulDet/PrimeVul

Cloning into 'PrimeVul'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 46 (delta 15), reused 8 (delta 8), pack-reused 23 (from 1)
Receiving objects: 100% (46/46), 39.22 KiB | 6.54 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [3]:
import os

# Check folder structure
os.listdir("PrimeVul")

['primevul_valid.jsonl',
 'openai_expr',
 'environment.yml',
 'LICENSE',
 'os_expr',
 'primevul_train.jsonl',
 'calc_vd_score.py',
 '.git',
 'README.md',
 'primevul_test.jsonl']

Load CodeBERT

In [28]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)   # MOVE MODEL TO GPU
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

Embedding Function

In [35]:
def get_function_embedding(code_snippet):
    inputs = tokenizer(
        code_snippet,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.squeeze().cpu().numpy()

Load Dataset - with limit

In [36]:
def load_dataset(file_path, limit=None):
    data = []
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            data.append(json.loads(line))
    return data


train_path = "PrimeVul/primevul_train.jsonl"  # adjust if needed

dataset = load_dataset(train_path, limit=None)  # LIMIT for Colab safety #limiting to 200 was selecting only [1]

print("Loaded samples:", len(dataset))

Loaded samples: 175797


Load Dataset - Without limit

In [30]:
def load_dataset(file_path):
    data = []
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_path = "PrimeVul/primevul_train.jsonl"  # adjust if needed
dataset = load_dataset(train_path)

print("Total samples:", len(dataset))

Total samples: 175797


Collect Balanced Data

In [37]:
balanced_data = []

count_0 = 0
count_1 = 0
limit_per_class = 100   # you can increase later

for item in dataset:
    label = item["target"]

    if label == 0 and count_0 < limit_per_class:
        balanced_data.append(item)
        count_0 += 1

    elif label == 1 and count_1 < limit_per_class:
        balanced_data.append(item)
        count_1 += 1

    # Stop when both classes collected
    if count_0 >= limit_per_class and count_1 >= limit_per_class:
        break

print("Collected:", len(balanced_data))
print("Class 0:", count_0, "Class 1:", count_1)

Collected: 200
Class 0: 100 Class 1: 100


Generate Embeddings (M1 Output)

In [38]:
embeddings = []
labels = []

for item in tqdm(balanced_data):
    code = item["func"]
    label = item["target"]

    emb = get_function_embedding(code)

    embeddings.append(emb)
    labels.append(label)

embeddings = np.array(embeddings)
labels = np.array(labels)

print("Embeddings shape:", embeddings.shape)
print("Label distribution:", dict(zip(*np.unique(labels, return_counts=True))))

100%|██████████| 200/200 [00:07<00:00, 26.11it/s]

Embeddings shape: (200, 768)
Label distribution: {np.int64(0): np.int64(100), np.int64(1): np.int64(100)}


Train-Test Split

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels   # IMPORTANT
)

Train Model

In [40]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', class_weight='balanced')

svm_model.fit(X_train, y_train)

SVC(class_weight='balanced', kernel='linear')

Evaluate Model

In [41]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = svm_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.875

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.90      0.88        20
           1       0.89      0.85      0.87        20

    accuracy                           0.88        40
   macro avg       0.88      0.88      0.87        40
weighted avg       0.88      0.88      0.87        40



In [42]:
code_example = """
int vulnerable(char *input) {
    char buffer[10];
    strcpy(buffer, input);
    return 0;
}
"""

emb = get_function_embedding(code_example)
prediction = svm_model.predict([emb])

print("Prediction:", prediction[0])
# 1 = vulnerable, 0 = safe

Prediction: 1
